In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from IPython.display import clear_output
import os
import time
import pandas as pd
import numpy as np
from datetime import date
from openai import OpenAI
from collections import Counter
import matplotlib.pyplot as plt
import re
%matplotlib inline

In [ ]:
# Put your OpenAI API key here
OPENAI_API_KEY = ...

# Specify the job title you are interested in
job_title_to_summarize = ...
# example: job_title_to_summarize = "data analyst"

# Provide alternative job titles to job_title_to_summarize, separated by a comma
job_title_alternatives = ...
# example: job_title_alternatives = "business intelligence, dashboard developer, data scientist, AI/ML researcher, ML ops / ML engineer, data engineer, data governance"

# Specify the country in which you want to search said jobs. You can choose between "ID" for Indonesia, "AU" for Australia, and "NZ" for New Zealand
country_of_interest = ...
# example: country_of_interest = "ID"

# Specify the number of web pages to search for. The higher the number, the more job postings the program will take but the longer time it will need
number_of_pages = ...
# example: number_of_pages = 10

In [ ]:
def scrape_job_posting_urls(country, job_keyword, pages):
    '''
    Scrapes job listings from seek.com or jobstreet based on the keyword and save the links to a parquet file
    country (string): country code from which the tool will scrape from. Right now, it supports Indonesia (ID), Australia (AU), and New Zealand (NZ)
    job_keyword (string): job keyword separated by space, e.g. 'sous chef', 'machine learning engineer'
    pages (int): maximum number of pages to scrape from
    '''
    options = Options()
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    
    keyword = "ref=search-standalone"
    
    pages = list(range(1, pages))
    page_no = iter(pages)
    
    links = []

    if country == "ID":
        home_url = "https://id.jobstreet.com/id/"
    elif country == "AU":
        home_url = "https://www.seek.com.au/"
    elif country == "NZ":
        home_url = "https://www.seek.co.nz/"
    while True:
        try:
            current_page = next(page_no)
            url_path = home_url + job_keyword.replace(' ','-') + "-jobs?page=" + str(current_page)
            clear_output(wait=True)
            print(f"Gettings links from {url_path}")
    
            # initializing web driver
            driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
            driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
                "source": """
                Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
                """
            })
            driver.get(url_path)
            time.sleep(5)
    
            # Get links in the current page and add them to the total list
            hrefs = []
            a_tags = driver.find_elements(By.TAG_NAME, "a")
            for i in range(len(a_tags)):
                try:
                    # Re-fetch the element to avoid staleness
                    a = driver.find_elements(By.TAG_NAME, "a")[i]
                    href = a.get_attribute("href")
                    if href and keyword in href:
                        hrefs.append(href)
                except:
                    continue
            
            links = links + hrefs
            time.sleep(5)
    
            # Close driver
            driver.quit()
            time.sleep(20)
    
            # Get out of the loop if there are no job listing links anymore
            if len(hrefs)==0:
                break
        except StopIteration:
            print("Iterator is exhausted.")
            break
    
    # Save links into a parquet file
    today = date.today()
    today_str = today.strftime('%Y%m%d')
    links_df = pd.DataFrame({'links': links, 'keyword': job_keyword, 'date': today})
    links_path = f'{country}_{job_keyword.replace(' ','-')}_job_listing_links_{today_str}.parquet'
    links_df.to_parquet(links_path)
    print(f"Job posting links saved into {links_path}")

# Step 1: Get All Job Listing Links For Data Analyst Postings

In [ ]:
# Specify the country, job title, and number of web pages to scrape from
country = job_title_to_summarize
job_keyword = country_of_interest
pages = number_of_pages

In [ ]:
# Run the job posting scraper - this will take a few minutes depending on the number of pages
scrape_job_posting_urls(country, job_keyword, pages)

# Step 2: Scrape Details From Each Job Listing

In [ ]:
# Specify the path to the folder in which you store the results from the previous step
link_path = ...

In [ ]:
# Get links from the 
link_files = []
for file in os.listdir(link_path):
    if 'job_listing_links' in file:
        link_files.append(file)

link_df = pd.DataFrame()

for i in link_files:
    df = pd.read_parquet(f'{link_path}/{i}')
    link_df = pd.concat([link_df,df])

link_df['id'] = link_df.apply(lambda x: x['links'].rsplit('=', 1)[-1] or x['links'], axis=1)
link_df.drop_duplicates(subset=['id'], keep='first', inplace=True)

link_df['job_title'] = None
link_df['advertiser'] = None
link_df['job_details'] = None

options = Options()
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

for index, row in link_df.iterrows():
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
        Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
        """
    })
    driver.get(row['links'])
    time.sleep(2)
    
    job_title = driver.find_elements(By.CSS_SELECTOR, '[data-automation="job-detail-title"]')
    for e in job_title:
        if e.is_displayed():
            link_df.at[index, 'job_title'] = e.text
    
    advertiser = driver.find_elements(By.CSS_SELECTOR, '[data-automation="advertiser-name"]')
    for e in advertiser:
        if e.is_displayed():
            link_df.at[index, 'advertiser'] = e.text
    
    details = driver.find_elements(By.CSS_SELECTOR, '[data-automation="jobAdDetails"]')
    job_details = ''
    for e in details:
        if e.is_displayed():
            job_details += e.text + '\n'
    link_df.at[index, 'job_details'] = job_details

    driver.quit()
    time.sleep(2)

link_df.to_parquet('batch_1_job_details.parquet')

# Step 3: Summarize key responsibilities, requirements, and good-to-haves

In [ ]:
jobs_df = pd.read_parquet('batch_1_job_details.parquet')

In [ ]:
# Drop jobs without job title, company, or details
jobs_df.dropna(subset=['job_title','advertiser','job_details'], inplace=True)

# Drop duplicate job listings, i.e. same job title coming from the same company
jobs_df.drop_duplicates(subset=['job_title','advertiser'], inplace=True)

In [ ]:
# Initialize OpenAI API client
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
jobs_df['summary'] = ''
i = 0
no_jobs = jobs_df.shape[0]

for index, row in jobs_df.iterrows():
    listing = row['job_title'] + '\n' + row['job_details']

    prompt = f'''You are an assistant tasked with collecting data for a research whose objective of finding out what are the education background and skillset required for {job_title_to_summarize} jobs. 
                Example of {job_title_to_summarize} jobs: {job_title_alternatives}. 
                The output will then be used to collate the most desired skills and/or education background for each job. 
                For your information, the end output will be insights like "for data scientists, the most desired education level is Masters followed by PhD, the most desired education background is data science followed by statistics, the most desired skillset is Python and Tensorflow, etc.."
                But for now, you are tasked to summarize the job listing details to enable such analysis.
                If it is not data related (for example: economist, financial analyst specializing in merger and acquisition, field engineer, butcher), just reply ''. 
                If it is data related, then do the following:
                1. Assign an appropriate job title that is the closest to the being advertised (including but not limited to the data related job examples listed previously).
                2. Summarize the details to get the key responsibilities, requirements, and work setting (hybrid, office, or remote) in a format exactly as the desired output sample below
                3. Make the key responsibilities generic (no more than 2 or 3 words), so instead of "analyze pattern in survey responses to deliver insights to business stakeholders", say "analyze survey results".
                4. For education level, write 'Bachelors' or 'Masters' or 'PhD'.
                5. For education background, write the discipline (e.g. 'Computer Science','Statistics','Data Science', etc.)
                6. For skill requirements, only write one or two words at most. If there are multiple requirements mentioned, write all of them. But if some of the skills are preferred, log them into the 'skill_preferred' list instead. Sample requirements: "Have experience creating models with TensorFlow, OpenCV; experience creating dashboards with Tableau, Power BI preferrred." The result should be: 'skill_requirements': ['TensorFlow','OpenCV'], 'skill_preferred': ['Tableau','Power BI']
                7. For experience, write the number of years of relevant experience required. If the job is for fresh graduates or no job experience required, put 0.
                Order of summary: job_title|key_responsibilities|education_level|education_background|skill_requirements|skill_preferred|experience|work_setting
                Sample desired output: data analyst|key_responsibilities:analyze patterns,create dashboards,create models|education_level:Bachelors,Masters|education_background:Statistics,Computer Science|skill_requirements:Tableau,Power BI,SQL,Python|skill_preferred:Excel,GCP,AWS,Snowflake|experience:5|work_setting:hybrid
                Now, summarize this job listing: {listing}
                '''
    response = client.responses.create(
        model="gpt-4.1",
        input=prompt
    )
    jobs_df.at[index, 'summary'] = response.output_text
    
    clear_output(wait=True) # Clear previous output, wait for new output
    i += 1
    print(f"{i} out of {no_jobs} summarized")

In [ ]:
jobs_df.to_parquet('batch_1_job_summary.parquet')

# Step 4: Count Job Requirement Recurrence and Visualize

In [ ]:
# Load parquet file that was saved earlier
jobs_df = pd.read_parquet('batch_1_job_summary.parquet')

In [ ]:
# Take out jobs that are not data related jobs
data_jobs = jobs_df[jobs_df['summary'].notna() & (jobs_df['summary'].str.strip() != "")]

In [ ]:
# Clean up the 
def extract_field(text, field):
    if pd.isna(text):
        return np.nan
    
    # Regex: look for "<field>... until | or end"
    pattern = rf"{field}(.*?)(?:\||$)"
    match = re.search(pattern, text)
    if match:
        return match.group(1).strip()
    else:
        return np.nan

def split_summary(df, col, fields):
    for f in fields:
        df[f] = df[col].apply(lambda x: extract_field(x, f))
    return df

# Example usage
cols = ['key_responsibilities','education_level',
        'education_background','skill_requirements','skill_preferred',
        'experience','work_setting']

data_jobs = split_summary(data_jobs, 'summary', cols)

data_jobs['skill_requirements_cleaned'] = data_jobs['skill_requirements'].apply(lambda x: str(x).replace(":",""))
skill_requirements = (
    data_jobs['skill_requirements_cleaned']
    .str.split(',')
    .explode()
    .str.strip()
)

# Remove 'None'
skill_requirements = skill_requirements[skill_requirements != 'None']

# Count occurrences
skill_requirements_count = dict(Counter(skill_requirements))
skill_requirements_count = sorted(skill_requirements_count.items(), key=lambda item: item[1], reverse=True)

Visualize top skills requirement by occurrence 

In [ ]:
data = skill_requirements_count
df = pd.DataFrame(data, columns=['Skill', 'Count'])

# Remove unwanted values and sort descending
remove_list = ['nan', 'null', 'na', '0']
df = df[~df['Skill'].str.lower().isin(remove_list)]
df = df.sort_values('Count', ascending=False)

# Pick top 20 only
top_n = 20
df = df.head(top_n)

# Visualize as a bar chart using matplotlib
plt.figure(figsize=(8,5))
plt.bar(df['Skill'], df['Count'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title(f"Top {top_n} Skills Frequency")
plt.xlabel("Skill")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

Visualize top education level requirement by occurrence 

In [ ]:
data = education_level_count
df = pd.DataFrame(data, columns=['Education Level', 'Count'])

# Remove unwanted values and sort descending
remove_list = ['nan', 'null', 'na', '0']
df = df[~df['Education Level'].str.lower().isin(remove_list)]
df = df.sort_values('Count', ascending=False)

# Pick top 10 only
top_n = 10
df = df.head(top_n)

# Visualize as a bar chart using matplotlib
plt.figure(figsize=(8,5))
plt.bar(df['Education Level'], df['Count'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title(f"Top {top_n} Skills Frequency")
plt.xlabel("Education Level")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

Visualize top education background requirement by occurrence 

In [ ]:
data = education_background_count
df = pd.DataFrame(data, columns=['Education Level', 'Count'])

# Remove unwanted values and sort descending
remove_list = ['nan', 'null', 'na', '0']
df = df[~df['Education Level'].str.lower().isin(remove_list)]
df = df.sort_values('Count', ascending=False)

# Pick top 10 only
top_n = 10
df = df.head(top_n)

# Visualize as a bar chart using matplotlib
plt.figure(figsize=(8,5))
plt.bar(df['Education Level'], df['Count'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title(f"Top {top_n} Skills Frequency")
plt.xlabel("Education Level")
plt.ylabel("Count")
plt.tight_layout()
plt.show()